# Walmart Store Sales Forecasting — PatchTST Baseline + MLflow Tracking

**გეგმა:**
1. მონაცემების ჩატვირთვა და feature engineering (Walmart Recruiting - Store Sales Forecasting ფორმატი: `train.csv`, `features.csv`, `stores.csv`)
2. WMAE მეტრიკის იმპლემენტაცია (holiday კვირებს 5x წონა)
3. დროზე დაფუძნებული train/val split
4. PatchTST baseline (`neuralforecast`)
5. Train/Val WMAE-ს ლოგირება MLflow-ში
6. ადგილი შემდეგი იტერაციებისთვის (hyperparams, feature-ები, TFT შედარება)

> **შენიშვნა:** სვეტების სახელები აღებულია სტანდარტული Kaggle Walmart კონკურსის ფორმატიდან. თუ შენი ფაილები სხვანაირად გამოიყურება, მითხარი და მოვარგებ.

## 0. Setup

In [1]:
# გავუშვათ პირველად, თუ გარემოში არ არის დაყენებული
%pip install -q neuralforecast mlflow pandas numpy scikit-learn


In [2]:
import os
import json
import numpy as np
import pandas as pd
import mlflow
import mlflow.pyfunc
from datetime import timedelta

from neuralforecast import NeuralForecast
from neuralforecast.models import PatchTST
from neuralforecast.losses.pytorch import MAE

pd.set_option("display.max_columns", 50)
np.random.seed(42)


In [3]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_PATH = '/content/drive/MyDrive/ML final project/ML_final'

if not os.path.exists(PROJECT_PATH):
    os.makedirs(PROJECT_PATH)
    print(f"it created: {PROJECT_PATH}")

%cd {PROJECT_PATH}

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/ML final project/ML_final


In [4]:
!pip install dagshub

In [5]:
import dagshub
dagshub.init(repo_owner='mesata', repo_name='Walmart---Store-Sales-Forecasting', mlflow=True)

Accessing as mkekn23

Initialized MLflow to track repo "mesata/Walmart---Store-Sales-Forecasting"

Repository mesata/Walmart---Store-Sales-Forecasting initialized!

In [6]:
TRAIN_PATH = "/content/drive/MyDrive/ML final project/walmart_data/train.csv.zip"
TEST_PATH = "/content/drive/MyDrive/ML final project/walmart_data/test.csv.zip"
FEATURES_PATH = "/content/drive/MyDrive/ML final project/walmart_data/features.csv.zip"
STORES_PATH = "/content/drive/MyDrive/ML final project/walmart_data/stores.csv"

VAL_WEEKS = 8          # ბოლო N კვირა გამოვიყენოთ ვალიდაციისთვის (time-based split)
HORIZON = VAL_WEEKS     # PatchTST-ის პროგნოზირების ჰორიზონტი ვალიდაციისთვის
INPUT_SIZE = 52         # რამდენი წარსული კვირა ვაძლევთ input-ად (baseline: 1 წელი)
MIN_SERIES_LENGTH = INPUT_SIZE + HORIZON + 4  # series ამის ქვემოთ გამოვრიცხოთ baseline-ისთვის



In [7]:
train = pd.read_csv("/content/drive/MyDrive/ML final project/walmart_data/train.csv.zip")

## 2. მონაცემების ჩატვირთვა და შერწყმა

In [8]:
train = pd.read_csv(TRAIN_PATH, parse_dates=["Date"])
features = pd.read_csv(FEATURES_PATH, parse_dates=["Date"])
stores = pd.read_csv(STORES_PATH)

# features.csv-ს ხშირად აქვს დუბლირებული IsHoliday სვეტი — მოვაშოროთ კონფლიქტი merge-ის წინ
if "IsHoliday" in features.columns:
    features = features.drop(columns=["IsHoliday"])

df = train.merge(features, on=["Store", "Date"], how="left")
df = df.merge(stores, on="Store", how="left")

markdown_cols = [c for c in df.columns if c.startswith("MarkDown")]
df[markdown_cols] = df[markdown_cols].fillna(0)

df = df.sort_values(["Store", "Dept", "Date"]).reset_index(drop=True)
print(df.shape)
df.head()


(421570, 16)


,Store,Dept,Date,Weekly_Sales,IsHoliday,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,Type,Size
0,1,1,2010-02-05,24924.50,False,42.31,2.572,0.0,0.0,0.0,0.0,0.0,211.096358,8.106,A,151315
1,1,1,2010-02-12,46039.49,True,38.51,2.548,0.0,0.0,0.0,0.0,0.0,211.242170,8.106,A,151315
2,1,1,2010-02-19,41595.55,False,39.93,2.514,0.0,0.0,0.0,0.0,0.0,211.289143,8.106,A,151315
3,1,1,2010-02-26,19403.54,False,46.63,2.561,0.0,0.0,0.0,0.0,0.0,211.319643,8.106,A,151315
4,1,1,2010-03-05,21827.90,False,46.50,2.625,0.0,0.0,0.0,0.0,0.0,211.350143,8.106,A,151315


## 3. WMAE მეტრიკა

Kaggle-ის ოფიციალური ფორმულა: holiday კვირებს 5x წონა აქვთ, დანარჩენებს — 1x.


In [9]:
def wmae(y_true: np.ndarray, y_pred: np.ndarray, is_holiday: np.ndarray) -> float:
    weights = np.where(is_holiday, 5.0, 1.0)
    return float(np.sum(weights * np.abs(y_true - y_pred)) / np.sum(weights))


## 4. Long-format მომზადება `neuralforecast`-ისთვის

`neuralforecast`-ს სჭირდება სვეტები: `unique_id`, `ds`, `y` (+ სურვილისამებრ `exogenous` ცვლადები).
თითოეული Store-Dept წყვილი — ცალკე დროითი მწკრივია (`unique_id`).


In [10]:
nf_df = df.copy()
nf_df["unique_id"] = nf_df["Store"].astype(str) + "_" + nf_df["Dept"].astype(str)
nf_df = nf_df.rename(columns={"Date": "ds", "Weekly_Sales": "y"})
nf_df["is_holiday_flag"] = nf_df["IsHoliday"].astype(int)

# ამოვიღოთ ძალიან მოკლე სერიები, baseline-ისთვის საკმარისი სიგრძით
series_lengths = nf_df.groupby("unique_id")["ds"].count()
valid_ids = series_lengths[series_lengths >= MIN_SERIES_LENGTH].index
nf_df = nf_df[nf_df["unique_id"].isin(valid_ids)].reset_index(drop=True)

nf_df = nf_df[["unique_id", "ds", "y", "is_holiday_flag"]].sort_values(["unique_id", "ds"]).reset_index(drop=True)

print(f"სერიების რაოდენობა (baseline): {nf_df['unique_id'].nunique()}")
print(f"სულ მწკრივები: {len(nf_df)}")
nf_df.head()


სერიების რაოდენობა (baseline): 2967
სულ მწკრივები: 415259


,unique_id,ds,y,is_holiday_flag
0,10_1,2010-02-05,40212.84,0
1,10_1,2010-02-12,67699.32,1
2,10_1,2010-02-19,49748.33,0
3,10_1,2010-02-26,33601.22,0
4,10_1,2010-03-05,36572.44,0


## 5. დროზე დაფუძნებული Train/Val split

ვალიდაცია — ბოლო `VAL_WEEKS` კვირა თითოეული სერიისთვის (არა შემთხვევითი split, რადგან დროითი მწკრივია).


In [11]:
max_date = nf_df["ds"].max()
val_cutoff = max_date - pd.Timedelta(weeks=VAL_WEEKS)

train_df = nf_df[nf_df["ds"] <= val_cutoff].copy()
val_df = nf_df[nf_df["ds"] > val_cutoff].copy()

# დავრწმუნდეთ, რომ ვალიდაციაში მხოლოდ ისეთი unique_id-ებია, რაც ტრენინგშიც გვხვდება
common_ids = set(train_df["unique_id"]) & set(val_df["unique_id"])
train_df = train_df[train_df["unique_id"].isin(common_ids)].reset_index(drop=True)
val_df = val_df[val_df["unique_id"].isin(common_ids)].reset_index(drop=True)

print(f"Train: {train_df['ds'].min().date()} → {train_df['ds'].max().date()}, {len(train_df)} rows")
print(f"Val:   {val_df['ds'].min().date()} → {val_df['ds'].max().date()}, {len(val_df)} rows")
print(f"საერთო სერიები: {len(common_ids)}")


Train: 2010-02-05 → 2012-08-31, 390346 rows
Val:   2012-09-07 → 2012-10-26, 23317 rows
საერთო სერიები: 2947


## 6. PatchTST Baseline

მინიმალური, სწრაფად გასატესტი კონფიგურაცია. შემდეგ ეტაპებზე დავამატებთ:
- exogenous features (markdown-ები, temperature, CPI, unemployment)
- longer input_size / hyperparameter tuning
- store/dept-level static features


In [12]:
BASELINE_PARAMS = dict(
    h=HORIZON,
    input_size=INPUT_SIZE,
    patch_len=16,
    stride=8,
    max_steps=200,       # baseline-ისთვის სწრაფი — მოგვიანებით გავზრდით
    batch_size=32,
    learning_rate=1e-3,
    loss=MAE(),
    random_seed=42,
    scaler_type="standard",
)

model = PatchTST(**BASELINE_PARAMS)
nf = NeuralForecast(models=[model], freq="W-FRI")  # Walmart მონაცემები კვირეულია (პარასკევი)

nf.fit(df=train_df[["unique_id", "ds", "y"]])


INFO:lightning_fabric.utilities.seed:Seed set to 42
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name                | Type              | Params | Mode 
------------------------------------------------------------------
0 | loss                | MAE               | 0      | train
1 | hist_cat_embeddings | ModuleList        | 0      | train
2 | futr_cat_embeddings | ModuleList        | 0      | train
3 | stat_cat_embeddings | ModuleList        | 0      | train
4 | padder_train        | ConstantPad1d     | 0      | train
5 | scaler              | TemporalNorm      | 0      | train
6 | model               | PatchTST_backbone | 406 K  | train
------------------------------------------------------------------
406 K     Trainable params
3 

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_steps=200` reached.


## 7. პროგნოზირება და WMAE გამოთვლა (Train + Val)

In [13]:
# --- Validation ---
val_forecast = nf.predict()  # აბრუნებს h ნაბიჯს training data-ს ბოლოდან
val_forecast = val_forecast.rename(columns={"PatchTST": "y_pred"})

val_eval = val_df.merge(val_forecast, on=["unique_id", "ds"], how="inner")
val_eval = val_eval.merge(
    nf_df[["unique_id", "ds", "is_holiday_flag"]],
    on=["unique_id", "ds"], how="left", suffixes=("", "_dup")
)

val_wmae = wmae(
    val_eval["y"].values,
    val_eval["y_pred"].values,
    val_eval["is_holiday_flag"].values.astype(bool),
)
print(f"Validation WMAE: {val_wmae:,.2f}  (n={len(val_eval)})")


INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Predicting: |          | 0/? [00:00<?, ?it/s]

Validation WMAE: 1,499.02  (n=23205)


In [14]:
# --- Train-ზე fit-ის ხარისხის შესამოწმებლად (in-sample, არა საბოლოო ხედვა) ---
# გამოვიყენოთ cross_validation ბოლო ფანჯარაზე ტრენინგ სეტშივე, რომ მივიღოთ შედარებადი "train WMAE"
train_cv = nf.cross_validation(
    df=train_df[["unique_id", "ds", "y"]],
    n_windows=1,
    step_size=HORIZON,
)
train_cv = train_cv.rename(columns={"PatchTST": "y_pred"})
train_cv = train_cv.merge(
    nf_df[["unique_id", "ds", "is_holiday_flag"]],
    on=["unique_id", "ds"], how="left"
)

train_wmae = wmae(
    train_cv["y"].values,
    train_cv["y_pred"].values,
    train_cv["is_holiday_flag"].values.astype(bool),
)
print(f"Train (last window CV) WMAE: {train_wmae:,.2f}  (n={len(train_cv)})")


INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name                | Type              | Params | Mode 
------------------------------------------------------------------
0 | loss                | MAE               | 0      | train
1 | hist_cat_embeddings | ModuleList        | 0      | train
2 | futr_cat_embeddings | ModuleList        | 0      | train
3 | stat_cat_embeddings | ModuleList        | 0      | train
4 | padder_train        | ConstantPad1d     | 0      | train
5 | scaler              | TemporalNorm      | 0      | train
6 | model               | PatchTST_backbone | 406 K  | train
------------------------------------------------------------------
406 K     Trainable params
3         Non-trainable params
406 K     Total params


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_steps=200` reached.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Predicting: |          | 0/? [00:00<?, ?it/s]

Train (last window CV) WMAE: 1,475.37  (n=23576)


## 8. MLflow ლოგირება

ყველაფერს ვახვევთ ერთ ფუნქციაში, რომ შემდეგი იტერაციები (feature-ები, hyperparams) ერთი ხაზის შეცვლით დავლოგოთ.


In [34]:
def run_pipeline(run_name: str, model_params: dict, notes: str = ""):
    """სრული pipeline: fit → predict → WMAE → MLflow log. აბრუნებს (train_wmae, val_wmae, nf)."""
    with mlflow.start_run(run_name=run_name):
        mlflow.log_params(model_params)
        mlflow.set_tag("notes", notes)
        mlflow.log_param("val_weeks", VAL_WEEKS)
        mlflow.log_param("input_size", model_params.get("input_size", INPUT_SIZE))
        mlflow.log_param("n_series_train", train_df["unique_id"].nunique())

        model = PatchTST(**model_params)
        nf_run = NeuralForecast(models=[model], freq="W-FRI")
        nf_run.fit(df=train_df[["unique_id", "ds", "y"]])

        # Validation WMAE
        val_fc = nf_run.predict().rename(columns={"PatchTST": "y_pred"})
        # val_df already contains 'is_holiday_flag', so merging with val_fc will preserve it.
        # The previous merge with nf_df was redundant and potentially problematic.
        val_ev = val_df.merge(val_fc, on=["unique_id", "ds"], how="inner")

        v_wmae = wmae(
            val_ev["y"].values,
            val_ev["y_pred"].values,
            val_ev["is_holiday_flag"].values.astype(bool),
        )

        # Train WMAE (last-window CV on train set)
        train_cv_run = nf_run.cross_validation(
            df=train_df[["unique_id", "ds", "y"]], n_windows=1, step_size=model_params["h"]
        ).rename(columns={"PatchTST": "y_pred"})
        # Merge with train_df to get 'is_holiday_flag' for the training subset
        train_cv_run = train_cv_run.merge(
            train_df[["unique_id", "ds", "is_holiday_flag"]], on=["unique_id", "ds"], how="left"
        )

        t_wmae = wmae(
            train_cv_run["y"].values, train_cv_run["y_pred"].values,
            train_cv_run["is_holiday_flag"].values.astype(bool)
        )

        mlflow.log_metric("train_wmae", t_wmae)
        mlflow.log_metric("val_wmae", v_wmae)

        print(f"[{run_name}] train_wmae={t_wmae:,.2f}  val_wmae={v_wmae:,.2f}")
        return t_wmae, v_wmae, nf_run


In [16]:
mlflow.set_experiment("PATCH TST")

2026/07/12 08:08:24 INFO mlflow.tracking.fluent: Experiment with name 'PATCH TST' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/5bc62234b85f46499d4bf36503df6fdd', creation_time=1783843704237, effective_trace_archival_retention=None, experiment_id='7', last_update_time=1783843704237, lifecycle_stage='active', name='PATCH TST', tags={}, trace_location=None, workspace='default'>

In [32]:
# Baseline run — ლოგირდება MLflow-ში
baseline_params = dict(
    h=HORIZON,
    input_size=INPUT_SIZE,
    patch_len=16,
    stride=8,
    max_steps=200,
    batch_size=32,
    learning_rate=1e-3,
    loss=MAE(),
    random_seed=42,
    scaler_type="standard",
)

train_wmae, val_wmae, nf_baseline = run_pipeline(
    run_name="patchtst_baseline",
    model_params=baseline_params,
    train_data_df=train_df, # Pass existing train_df for baseline
    val_data_df=val_df,     # Pass existing val_df for baseline
    notes="baseline: მხოლოდ y, გარეშე exogenous features, input_size=52, max_steps=200",
)

INFO:lightning_fabric.utilities.seed:Seed set to 42
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name                | Type              | Params | Mode 
------------------------------------------------------------------
0 | loss                | MAE               | 0      | train
1 | hist_cat_embeddings | ModuleList        | 0      | train
2 | futr_cat_embeddings | ModuleList        | 0      | train
3 | stat_cat_embeddings | ModuleList        | 0      | train
4 | padder_train        | ConstantPad1d     | 0      | train
5 | scaler              | TemporalNorm      | 0      | train
6 | model               | PatchTST_backbone | 406 K  | train
------------------------------------------------------------------
406 K     Trainable params
3 

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_steps=200` reached.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Predicting: |          | 0/? [00:00<?, ?it/s]

[patchtst_baseline] train_wmae=1,475.37  val_wmae=1,499.02
🏃 View run patchtst_baseline at: https://dagshub.com/mesata/Walmart---Store-Sales-Forecasting.mlflow/#/experiments/7/runs/da21e88fa923465eb6120ba3a0a2dabc
🧪 View experiment at: https://dagshub.com/mesata/Walmart---Store-Sales-Forecasting.mlflow/#/experiments/7


In [18]:
with mlflow.start_run(run_name="naive_baselines"):
    mlflow.set_tag("notes", "sanity-check baselines: last-value carry-forward და seasonal t-52")

    # --- Naive: last observed value carried forward ---
    last_vals = train_df.sort_values("ds").groupby("unique_id")["y"].last()
    naive_last_pred = val_df["unique_id"].map(last_vals)
    naive_last_wmae = wmae(
        val_df["y"].values,
        naive_last_pred.values,
        val_df["is_holiday_flag"].values.astype(bool),
    )

    # --- Naive: seasonal (იგივე კვირა, 1 წლის წინ = t-52) ---
    lookup = nf_df.set_index(["unique_id", "ds"])["y"]
    seasonal_ds = val_df["ds"] - pd.Timedelta(weeks=52)
    seasonal_keys = list(zip(val_df["unique_id"], seasonal_ds))
    naive_seasonal_pred = pd.Series(
        [lookup.get(k, np.nan) for k in seasonal_keys], index=val_df.index
    )
    coverage = naive_seasonal_pred.notna().mean()
    mask = naive_seasonal_pred.notna()

    naive_seasonal_wmae = wmae(
        val_df.loc[mask, "y"].values,
        naive_seasonal_pred[mask].values,
        val_df.loc[mask, "is_holiday_flag"].values.astype(bool),
    )

    mlflow.log_metric("naive_last_val_wmae", naive_last_wmae)
    mlflow.log_metric("naive_seasonal_val_wmae", naive_seasonal_wmae)
    mlflow.log_metric("naive_seasonal_coverage", coverage)

    print(f"Naive (last value)     val_wmae = {naive_last_wmae:,.2f}")
    print(f"Naive (seasonal t-52)  val_wmae = {naive_seasonal_wmae:,.2f}  (coverage: {coverage:.1%} სერიების)")
    print(f"PatchTST baseline      val_wmae = {val_wmae:,.2f}")

    if val_wmae < min(naive_last_wmae, naive_seasonal_wmae):
        print("\n✓ PatchTST-მა აჯობა ორივე naive baseline-საც — საწყისი დადებითი ანალიზის საფუძველი არსებობს.")
    else:
        print("\n⚠ PatchTST ვერ აჯობებს რომელიმე naive baseline-ს — ღირს exogenous features/feature engineering-ის საკითხის განხილვა.")

Naive (last value)     val_wmae = 2,415.07
Naive (seasonal t-52)  val_wmae = 1,718.32  (coverage: 99.2% სერიების)
PatchTST baseline      val_wmae = 1,499.02

✓ PatchTST-მა აჯობა ორივე naive baseline-საც — საწყისი დადებითი ანალიზის საფუძველი არსებობს.
🏃 View run naive_baselines at: https://dagshub.com/mesata/Walmart---Store-Sales-Forecasting.mlflow/#/experiments/7/runs/0c92627d3dd54a0fb7e2840a8c1b9880
🧪 View experiment at: https://dagshub.com/mesata/Walmart---Store-Sales-Forecasting.mlflow/#/experiments/7


In [19]:
nf_df = df.copy()
nf_df["unique_id"] = nf_df["Store"].astype(str) + "_" + nf_df["Dept"].astype(str)
nf_df = nf_df.rename(columns={"Date": "ds", "Weekly_Sales": "y"})
nf_df["is_holiday_flag"] = nf_df["IsHoliday"].astype(int)

# ეს ცვლადები ცნობილია მომავალშიც (features.csv Kaggle-ის ორიგინალში მოიცავს ტესტ პერიოდსაც) —
# ამიტომ PatchTST-ში futr_exog_list-ად გამოვიყენებთ
EXOG_COLS = ["is_holiday_flag", "Temperature", "Fuel_Price", "CPI", "Unemployment"] + markdown_cols

nf_df[EXOG_COLS] = nf_df[EXOG_COLS].fillna(0)

series_lengths = nf_df.groupby("unique_id")["ds"].count()
valid_ids = series_lengths[series_lengths >= MIN_SERIES_LENGTH].index
nf_df = nf_df[nf_df["unique_id"].isin(valid_ids)].reset_index(drop=True)

nf_df = nf_df[["unique_id", "ds", "y"] + EXOG_COLS].sort_values(["unique_id", "ds"]).reset_index(drop=True)

print(f"სერიების რაოდენობა: {nf_df['unique_id'].nunique()}")
print(f"Exogenous features: {EXOG_COLS}")
nf_df.head()

სერიების რაოდენობა: 2967
Exogenous features: ['is_holiday_flag', 'Temperature', 'Fuel_Price', 'CPI', 'Unemployment', 'MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5']


,unique_id,ds,y,is_holiday_flag,Temperature,Fuel_Price,CPI,Unemployment,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5
0,10_1,2010-02-05,40212.84,0,54.34,2.962,126.442065,9.765,0.0,0.0,0.0,0.0,0.0
1,10_1,2010-02-12,67699.32,1,49.96,2.828,126.496258,9.765,0.0,0.0,0.0,0.0,0.0
2,10_1,2010-02-19,49748.33,0,58.22,2.915,126.526286,9.765,0.0,0.0,0.0,0.0,0.0
3,10_1,2010-02-26,33601.22,0,52.77,2.825,126.552286,9.765,0.0,0.0,0.0,0.0,0.0
4,10_1,2010-03-05,36572.44,0,55.92,2.877,126.578286,9.765,0.0,0.0,0.0,0.0,0.0


In [52]:
def run_pipeline(run_name: str, model_params: dict, val_size: int = 0, notes: str = ""):
    """სრული pipeline: fit → predict → WMAE → MLflow log.
    ავტომატურად გამორიცხავს სერიებს, რომლებსაც input_size + val_size-ისთვის საკმარისი სიგრძე არ აქვთ."""
    feature_cols = ["unique_id", "ds", "y"]
    input_size = model_params["input_size"]
    min_len_needed = input_size + val_size

    train_counts = train_df.groupby("unique_id")["ds"].count()
    ok_ids = train_counts[train_counts >= min_len_needed].index
    n_dropped = train_df["unique_id"].nunique() - len(ok_ids)

    train_sub = train_df[train_df["unique_id"].isin(ok_ids)]
    val_sub = val_df[val_df["unique_id"].isin(ok_ids)]

    if len(ok_ids) == 0:
        max_available = train_counts.max()
        raise ValueError(
            f"[{run_name}] ვერცერთი სერია ვერ აკმაყოფილებს input_size+val_size="
            f"{min_len_needed} მოთხოვნას. ყველაზე გრძელ სერიას მხოლოდ {max_available} "
            f"კვირა აქვს train-ში. შეამცირე input_size."
        )
    with mlflow.start_run(run_name=run_name):
        log_params = {k: v for k, v in model_params.items() if k != "loss"}
        log_params["loss"] = type(model_params["loss"]).__name__
        mlflow.log_params(log_params)
        mlflow.log_param("internal_val_size", val_size)
        mlflow.set_tag("notes", notes)
        mlflow.log_param("val_weeks", VAL_WEEKS)
        mlflow.log_param("n_series_train", len(ok_ids))
        mlflow.log_param("n_series_dropped_too_short", n_dropped)

        model = PatchTST(**model_params)
        nf_run = NeuralForecast(models=[model], freq="W-FRI")

        fit_kwargs = {"df": train_sub[feature_cols]}
        if val_size > 0:
            fit_kwargs["val_size"] = val_size
        nf_run.fit(**fit_kwargs)

        # --- Validation WMAE ---
        val_fc = nf_run.predict().rename(columns={"PatchTST": "y_pred"})
        val_ev = val_sub.merge(val_fc, on=["unique_id", "ds"], how="inner")
        v_wmae = wmae(val_ev["y"].values, val_ev["y_pred"].values, val_ev["is_holiday_flag"].values.astype(bool))

        # --- Train WMAE (last-window CV) ---
        train_cv_run = nf_run.cross_validation(
            df=train_sub[feature_cols], n_windows=1, step_size=model_params["h"]
        ).rename(columns={"PatchTST": "y_pred"})
        train_cv_run = train_cv_run.merge(
            nf_df[["unique_id", "ds", "is_holiday_flag"]], on=["unique_id", "ds"], how="left"
        )
        t_wmae = wmae(
            train_cv_run["y"].values, train_cv_run["y_pred"].values,
            train_cv_run["is_holiday_flag"].values.astype(bool)
        )

        mlflow.log_metric("train_wmae", t_wmae)
        mlflow.log_metric("val_wmae", v_wmae)

        print(f"[{run_name}] train_wmae={t_wmae:,.2f}  val_wmae={v_wmae:,.2f}  "
              f"(series: {len(ok_ids)}, dropped: {n_dropped})")
        return t_wmae, v_wmae, nf_run

In [50]:
EARLY_STOP_VAL_SIZE = HORIZON  # internal validation carve-out ტრენინგ სეტის ბოლოდან

configs = {
    "patchtst_steps_300": dict(
        h=HORIZON, input_size=INPUT_SIZE,
        patch_len=16, stride=8,
        max_steps=300,
        batch_size=32, learning_rate=1e-3,
        loss=MAE(), random_seed=42, scaler_type="standard",
    ),
    "patchtst_steps_600": dict(
        h=HORIZON, input_size=INPUT_SIZE,
        patch_len=16, stride=8,
        max_steps=600,
        batch_size=32, learning_rate=1e-3,
        loss=MAE(), random_seed=42, scaler_type="standard",
    ),
    "patchtst_longer_input": dict(
        h=HORIZON, input_size=104,
        patch_len=16, stride=8,
        max_steps=600,
        batch_size=32, learning_rate=1e-3,
        loss=MAE(), random_seed=42, scaler_type="standard",
    ),
    "patchtst_finer_patches": dict(
        h=HORIZON, input_size=104,
        patch_len=8, stride=4,
        max_steps=600,
        batch_size=32, learning_rate=5e-4,
        loss=MAE(), random_seed=42, scaler_type="standard",
    ),
}

results = {}
for run_name, params in configs.items():
    t_wmae, v_wmae, _ = run_pipeline(
        run_name=run_name,
        model_params=params,
        val_size=0,   # <-- Lightning-ის internal early stopping აღარ გვჭირდება
        notes=f"fixed-steps sweep: input_size={params['input_size']}, patch_len={params['patch_len']}, "
              f"stride={params['stride']}, max_steps={params['max_steps']}",
    )
    results[run_name] = (t_wmae, v_wmae)

print("\n=== შედარება ===")
print(f"{'run':<28}{'train_wmae':>12}{'val_wmae':>12}")
print(f"{'patchtst_baseline':<28}{train_wmae:>12,.2f}{val_wmae:>12,.2f}")
for run_name, (t, v) in results.items():
    print(f"{run_name:<28}{t:>12,.2f}{v:>12,.2f}")
results = {}
for run_name, params in configs.items():
    uses_early_stop = "early_stop_patience_steps" in params
    t_wmae, v_wmae, _ = run_pipeline(
        run_name=run_name,
        model_params=params,
        val_size=EARLY_STOP_VAL_SIZE if uses_early_stop else 0,
        notes=f"hyperparam sweep: input_size={params['input_size']}, patch_len={params['patch_len']}, "
              f"stride={params['stride']}, max_steps={params['max_steps']}, "
              f"early_stop_patience={params.get('early_stop_patience_steps', 'off')}",
    )
    results[run_name] = (t_wmae, v_wmae)

print("\n=== შედარება ===")
print(f"{'run':<28}{'train_wmae':>12}{'val_wmae':>12}")
print(f"{'patchtst_baseline':<28}{train_wmae:>12,.2f}{val_wmae:>12,.2f}")
for run_name, (t, v) in results.items():
    print(f"{run_name:<28}{t:>12,.2f}{v:>12,.2f}")

INFO:lightning_fabric.utilities.seed:Seed set to 42
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name                | Type              | Params | Mode 
------------------------------------------------------------------
0 | loss                | MAE               | 0      | train
1 | hist_cat_embeddings | ModuleList        | 0      | train
2 | futr_cat_embeddings | ModuleList        | 0      | train
3 | stat_cat_embeddings | ModuleList        | 0      | train
4 | padder_train        | ConstantPad1d     | 0      | train
5 | scaler              | TemporalNorm      | 0      | train
6 | model               | PatchTST_backbone | 406 K  | train
------------------------------------------------------------------
406 K     Trainable params
3 

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_steps=600` reached.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Predicting: |          | 0/? [00:00<?, ?it/s]

[patchtst_finer_patches] train_wmae=1,443.30  val_wmae=1,402.16  (series: 2846, dropped: 101)
🏃 View run patchtst_finer_patches at: https://dagshub.com/mesata/Walmart---Store-Sales-Forecasting.mlflow/#/experiments/7/runs/d216e2b8bb8d4bf6ad74f551b9b62e41
🧪 View experiment at: https://dagshub.com/mesata/Walmart---Store-Sales-Forecasting.mlflow/#/experiments/7

=== შედარება ===
run                           train_wmae    val_wmae
patchtst_baseline               1,475.37    1,499.02
patchtst_steps_300              1,765.81    1,546.68
patchtst_steps_600              1,710.08    1,442.01
patchtst_longer_input           1,467.37    1,417.38
patchtst_finer_patches          1,443.30    1,402.16


In [53]:
configs_round2 = {
    # "patchtst_finer_more_steps": dict(
    #     h=HORIZON, input_size=104,
    #     patch_len=8, stride=4,
    #     max_steps=1000,                 # 600 → 1000, ვამოწმებთ თუ კიდევ იმჯობინებს
    #     batch_size=32, learning_rate=5e-4,
    #     loss=MAE(), random_seed=42, scaler_type="standard",
    # ),
    # "patchtst_even_finer": dict(
    #     h=HORIZON, input_size=104,
    #     patch_len=4, stride=2,          # კიდევ უფრო წვრილი patches
    #     max_steps=600,
    #     batch_size=32, learning_rate=5e-4,
    #     loss=MAE(), random_seed=42, scaler_type="standard",
    # ),
   "patchtst_longer_history": dict(
    h=HORIZON, input_size=120,      # 156 → 120 (train-ში მაქსიმუმ ~135 კვირაა ხელმისაწვდომი)
    patch_len=8, stride=4,
    max_steps=600,
    batch_size=32, learning_rate=5e-4,
    loss=MAE(), random_seed=42, scaler_type="standard",
),
    "patchtst_bigger_model": dict(
        h=HORIZON, input_size=104,
        patch_len=8, stride=4,
        hidden_size=256,                # default-ზე მეტი capacity (ხშირად 128 default)
        n_heads=8,
        max_steps=600,
        batch_size=32, learning_rate=5e-4,
        loss=MAE(), random_seed=42, scaler_type="standard",
    ),
}

results_round2 = {}
for run_name, params in configs_round2.items():
    t_wmae, v_wmae, _ = run_pipeline(
        run_name=run_name,
        model_params=params,
        val_size=0,
        notes=f"round 2 (finer_patches-ზე დაფუძნებული): input_size={params['input_size']}, "
              f"patch_len={params['patch_len']}, stride={params['stride']}, max_steps={params['max_steps']}",
    )
    results_round2[run_name] = (t_wmae, v_wmae)

print("\n=== Round 2 შედარება (საუკეთესო წინა: patchtst_finer_patches = 1,402.16) ===")
print(f"{'run':<28}{'train_wmae':>12}{'val_wmae':>12}")
for run_name, (t, v) in results_round2.items():
    print(f"{run_name:<28}{t:>12,.2f}{v:>12,.2f}")

INFO:lightning_fabric.utilities.seed:Seed set to 42
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name                | Type              | Params | Mode 
------------------------------------------------------------------
0 | loss                | MAE               | 0      | train
1 | hist_cat_embeddings | ModuleList        | 0      | train
2 | futr_cat_embeddings | ModuleList        | 0      | train
3 | stat_cat_embeddings | ModuleList        | 0      | train
4 | padder_train        | ConstantPad1d     | 0      | train
5 | scaler              | TemporalNorm      | 0      | train
6 | model               | PatchTST_backbone | 433 K  | train
------------------------------------------------------------------
433 K     Trainable params
3 

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_steps=600` reached.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Predicting: |          | 0/? [00:00<?, ?it/s]

[patchtst_bigger_model] train_wmae=1,614.71  val_wmae=1,423.95  (series: 2846, dropped: 101)
🏃 View run patchtst_bigger_model at: https://dagshub.com/mesata/Walmart---Store-Sales-Forecasting.mlflow/#/experiments/7/runs/8d52c41a00f447989bcfbf51e8d79a1e
🧪 View experiment at: https://dagshub.com/mesata/Walmart---Store-Sales-Forecasting.mlflow/#/experiments/7

=== Round 2 შედარება (საუკეთესო წინა: patchtst_finer_patches = 1,402.16) ===
run                           train_wmae    val_wmae
patchtst_longer_history         1,621.78    1,385.01
patchtst_bigger_model           1,614.71    1,423.95


In [ ]:
configs_round2 = {
"patchtst_finer_patches_more_steps": dict(
    h=HORIZON, input_size=104,
    patch_len=4, stride=2,          # even_finer-ის structural ცვლილება
    max_steps=1000,                 # finer_more_steps-ის ტრენინგის ხანგრძლივობა
    batch_size=32, learning_rate=5e-4,
    loss=MAE(), random_seed=42, scaler_type="standard",
),

"patchtst_longer_history_more_steps": dict(
    h=HORIZON, input_size=120,      # longer_history-ის structural ცვლილება
    patch_len=8, stride=4,
    max_steps=1000,                 # ისევ მეტი ნაბიჯი
    batch_size=32, learning_rate=5e-4,
    loss=MAE(), random_seed=42, scaler_type="standard",
),
}

results_round3 = {}
for run_name, params in configs_round2.items():
    t_wmae, v_wmae, _ = run_pipeline(
        run_name=run_name,
        model_params=params,
        val_size=0,
        notes=f"round 2 (finer_patches-ზე დაფუძნებული): input_size={params['input_size']}, "
              f"patch_len={params['patch_len']}, stride={params['stride']}, max_steps={params['max_steps']}",
    )
    results_round3[run_name] = (t_wmae, v_wmae)


print(f"{'run':<28}{'train_wmae':>12}{'val_wmae':>12}")
for run_name, (t, v) in results_round3.items():
    print(f"{run_name:<28}{t:>12,.2f}{v:>12,.2f}")

INFO:lightning_fabric.utilities.seed:Seed set to 42
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name                | Type              | Params | Mode 
------------------------------------------------------------------
0 | loss                | MAE               | 0      | train
1 | hist_cat_embeddings | ModuleList        | 0      | train
2 | futr_cat_embeddings | ModuleList        | 0      | train
3 | stat_cat_embeddings | ModuleList        | 0      | train
4 | padder_train        | ConstantPad1d     | 0      | train
5 | scaler              | TemporalNorm      | 0      | train
6 | model               | PatchTST_backbone | 457 K  | train
------------------------------------------------------------------
457 K     Trainable params
3 

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_steps=1000` reached.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Predicting: |          | 0/? [00:00<?, ?it/s]

[patchtst_longer_history_more_steps] train_wmae=1,555.88  val_wmae=1,516.60  (series: 2779, dropped: 168)
🏃 View run patchtst_longer_history_more_steps at: https://dagshub.com/mesata/Walmart---Store-Sales-Forecasting.mlflow/#/experiments/7/runs/f9b398410cc54a04bad8c5b533931d9d
🧪 View experiment at: https://dagshub.com/mesata/Walmart---Store-Sales-Forecasting.mlflow/#/experiments/7
run                           train_wmae    val_wmae
patchtst_finer_patches_more_steps    1,476.02    1,432.64
patchtst_longer_history_more_steps    1,555.88    1,516.60
